# Bring your own Container with Azure Container Apps Sandboxes

## Prerequisites
- An Azure subscription with the **Container Apps SandboxGroup Data Owner** role assigned at the resource group or sandbox group scope.
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively) (`az login`). 
- The `azure-containerapps-sandbox` SDK installed in this notebook's Python environment.

Click **Run All** to execute the full lifecycle, or step through each cell to inspect outputs along the way.


In [ ]:
# Install the SDK and supporting packages (skip if already installed)
%pip install -q --upgrade azure-containerapps-sandbox azure-identity azure-mgmt-resource azure-mgmt-authorization

### 0. Initialize variables and clients

Adjust `resource_group_name`, `sandbox_group_name`, and `location` if you want to target existing resources or a different region.


In [ ]:
import json, subprocess, sys, time, uuid, urllib.request
from pathlib import Path
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.containerapps.sandbox import SandboxGroupManagementClient, SandboxGroupClient, endpoint_for_region

# ── Configuration ─────────────────────────────────────────────────────────
lab_name                = Path(globals().get('__vsc_ipynb_file__') or '01-getting-started.ipynb').stem
resource_group_name     = 'lab-aca-sandboxes'     # change if using existing Resource Group
sandbox_group_name      = 'byoc-lab'              # change if using existing Sandbox Group
location                = 'westus3'               # change to your preferred Azure Location
lab_labels              = {'lab': lab_name}       # tag every resource so cleanup can find them

# ── Get current subscription info from Azure CLI (run `az login` first) ───
try:
    proc = subprocess.run('az account show -o json',
        capture_output=True, text=True, check=True, shell=True)
    subscription = json.loads(proc.stdout)
    subscription_id = subscription['id']
except subprocess.CalledProcessError as e:
    sys.exit(f'❌ Azure CLI not installed or not signed in. Run `az login` first.\n{e.stderr}')

cli_credential = AzureCliCredential() # reused by every Azure client below
resource_mgmt_client = ResourceManagementClient(cli_credential, subscription_id) # manage resource groups
sandboxgroup_mgmt_client = SandboxGroupManagementClient(cli_credential, subscription_id=subscription_id, resource_group=resource_group_name) # manage sandbox groups

print(f'🧪 Lab:                   {lab_name}')
print(f'👤 User:                  {subscription["user"]["name"]}')
print(f'🏛️ Tenant Id:             {subscription["tenantId"]}')
print(f'🔑 Subscription:          {subscription["name"]} ({subscription_id})')
print(f'🌍 Target Azure Region:   {location}')
print(f'📁 Target Resource Group: {resource_group_name}')
print(f'📦 Target Sandbox Group:  {sandbox_group_name}')


### 1. Create the resource group and sandbox group

A **sandbox group** is the top-level management boundary — all sandboxes, disk
images, snapshots, volumes, and secrets are scoped to it. Use sandbox groups to
organize compute by application, team, or environment.

This cell creates (or reuses) both the Azure resource group and the sandbox
group inside it. The sandbox group is an ARM resource, so it goes through the
control plane (`management.azure.com`).


In [ ]:
if resource_mgmt_client.resource_groups.check_existence(resource_group_name):
    resource_group = resource_mgmt_client.resource_groups.get(resource_group_name)
    print(f'♻️ Using existing resource group: {resource_group.name} ({resource_group.location})')
else:
    resource_group = resource_mgmt_client.resource_groups.create_or_update(
        resource_group_name, {'location': location, 'tags': lab_labels})
    print(f'✅ Created resource group: {resource_group.name} ({resource_group.location})')

# Reuse the sandbox group if it exists; otherwise create it tagged with the lab name.
existing = next((g for g in sandboxgroup_mgmt_client.list_groups() if g.name == sandbox_group_name), None)
if existing:
    action, icon = 'Using existing', '♻️'
else:
    sandboxgroup_mgmt_client.create_group(
        sandbox_group_name, location=resource_group.location, tags=lab_labels)
    action, icon = 'Created', '✅'

# Refetch to get a fully-hydrated object regardless of which branch ran above.
sandboxgroup = sandboxgroup_mgmt_client.get_group(sandbox_group_name)

sandbox_group_url = f'https://sandboxes.azure.com/sandbox-groups/{subscription_id}/{resource_group_name}/{sandbox_group_name}/sandboxes'
print(f'{icon} {action} sandbox group: {sandboxgroup.name}')
print(f'  🆔 Id:                   {sandboxgroup.id}')
print(f'  🌍 Location:             {sandboxgroup.location}')
print(f'  ⚙️ Provisioning state:   {sandboxgroup.properties["provisioningState"]}')
print(f'  🏷️ Tags:                 {sandboxgroup.tags or "{}"}')
print(f'  🔗 Portal:               {sandbox_group_url}')

# Build the data-plane client now that we know the group's region — used from step 5 onward
# to create disk images and sandboxes (sandboxes.azure.com, regional ADC endpoint).
client = SandboxGroupClient(
    endpoint_for_region(sandboxgroup.location), cli_credential,
    subscription_id=subscription_id, resource_group=resource_group_name,
    sandbox_group=sandbox_group_name)
print(f'  🔌 Data-plane endpoint:  {endpoint_for_region(sandboxgroup.location)}')



### 2. Create an Azure Container Registry

To bring your own container, you first need somewhere to store the image. We create an **Azure Container Registry (ACR)** in the same resource group as the sandbox group using the Azure CLI.

Later steps build a sample image directly in this registry (no local Docker required) and then convert it into a **private disk image** that sandboxes can boot from.


In [ ]:
# Create an Azure Container Registry (ACR) in the same resource group via the Azure CLI.
# Registry names are globally unique, alphanumeric, and 5-50 characters long.
acr_name = f'byoc{uuid.uuid4().hex[:12]}'
acr_tags = ' '.join(f'{k}={v}' for k, v in lab_labels.items())

proc = subprocess.run(
    f'az acr create --resource-group {resource_group_name} --name {acr_name} '
    f'--sku Basic --location {location} --tags {acr_tags} -o json',
    capture_output=True, text=True, check=True, shell=True)
acr = json.loads(proc.stdout)
acr_id = acr['id']
acr_login_server = acr['loginServer']

print(f'✅ Created Azure Container Registry: {acr_name}')
print(f'  🆔 Id:           {acr_id}')
print(f'  🌐 Login server: {acr_login_server}')
print(f'  📦 SKU:          {acr["sku"]["name"]}')
print(f'  🌍 Location:     {acr["location"]}')


### 3. Get registry credentials

Pulling from a **private** registry requires credentials. The disk-image build authenticates with an explicit username/token pair (`RegistryCredentials`) rather than a managed identity, so we mint a **short-lived ACR access token** with `az acr login --expose-token`.

This needs no admin user and no stored password — the token is derived from your current `az login` session and paired with the well-known all-zeros GUID username. We capture it here and hand it to the disk-image build in step 5.


In [ ]:
from azure.containerapps.sandbox import RegistryCredentials

# Mint a short-lived ACR access token from the current `az login` session.
# `--expose-token` returns an AAD-backed token used as the password; the username
# is always the well-known all-zeros GUID. No admin user or stored secret required.
proc = subprocess.run(
    f'az acr login --name {acr_name} --expose-token -o json',
    capture_output=True, text=True, check=True, shell=True)
acr_token = json.loads(proc.stdout)['accessToken']
registry_credentials = RegistryCredentials(
    username='00000000-0000-0000-0000-000000000000', token=acr_token)
print(f'🔑 Obtained ACR access token for {acr_login_server} (length={len(acr_token)})')


### 4. Build an image in the cloud with ACR Tasks

`az acr build` runs the Docker build **inside Azure** using ACR Tasks — so you don't need Docker installed locally and you don't need any source files on this machine. We point it at a public Git repository as the build context; ACR clones it, builds the `Dockerfile`, and pushes the resulting image into the registry.

Here we build the small `acr-build-helloworld-node` sample. Swap `build_context` for your own repo (or a local folder) to bring your own application.


In [ ]:
# Build a sample image in the cloud with ACR Tasks — no local Docker or source files needed.
# The build context is a public Git repo; ACR clones it, builds the Dockerfile, and pushes the result.
image_tag = 'helloworld:v1'
build_context = 'https://github.com/Azure-Samples/acr-build-helloworld-node.git'

print(f'🏗️ Running cloud build for {acr_login_server}/{image_tag} (this can take a few minutes)...')
proc = subprocess.run(
    f'az acr build --registry {acr_name} --image {image_tag} {build_context} -o json',
    capture_output=True, text=True, shell=True)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
print(f'✅ Built and pushed image: {acr_login_server}/{image_tag}')

# Confirm the repository now exists in the registry.
proc = subprocess.run(f'az acr repository list --name {acr_name} -o json',
    capture_output=True, text=True, check=True, shell=True)
print(f'📚 Repositories in {acr_name}: {json.loads(proc.stdout)}')


### 5. Create a private disk image from the registry image

A sandbox boots from a **disk image** — an OCI container image converted into a root filesystem. `begin_create_disk_image(<image-ref>, name=..., registry_credentials=...)` pulls the image from ACR and converts it, returning an `LROPoller` that we wait on until the image is `Ready`.

Because the registry is private, we pass the `RegistryCredentials` we obtained in step 3. After creating the image we call `list_disk_images()` to confirm it shows up among the group's private disk images.

> The first build can take several minutes while the image is pulled and converted.


In [ ]:
# Create a private disk image from the image we pushed to ACR.
# The build pulls the image using the ACR token captured in step 3 (registry_credentials).
# Alternative: pass managed_identity_resource_id=... to pull with a user-assigned identity.
byoc_image_ref = f'{acr_login_server}/{image_tag}'
disk_image = client.begin_create_disk_image(
    byoc_image_ref, name='byoc-helloworld',
    registry_credentials=registry_credentials, polling_timeout=900).result()
print(f'💾 Created disk image: {disk_image.id}')
print(f'  ⚙️ State:  {disk_image.status.state if disk_image.status else "?"}')
print(f'  💿 Source: {disk_image.image.base if disk_image.image else "?"}')

# List private disk images and confirm ours is present.
private_disk_images = list(client.list_disk_images())
print(f'\n📚 Private disk images in the group ({len(private_disk_images)}):')
found = False
for img in private_disk_images:
    marker = '  <-- just created' if img.id == disk_image.id else ''
    print(f'  - {img.id}  name={img.name or img.labels.get("name", "")}{marker}')
    found = found or img.id == disk_image.id
assert found, 'Expected the newly created disk image to appear in list_disk_images()'
print('✅ Confirmed the disk image is in the private disk image list')


### 6. Launch a sandbox from your image

Finally, boot a sandbox straight from the private disk image with `begin_create_sandbox(disk_id=...)`. The sandbox starts with your image as its root filesystem.

To prove the bring-your-own-container image actually booted, we run a command that only works on *our* image: the default sandbox disk is plain Ubuntu, but our image bakes in **Node.js**, so `node --version` succeeding confirms the custom image is live.


In [ ]:
# Launch a sandbox directly from our private disk image.
sandbox = client.begin_create_sandbox(disk_id=disk_image.id, labels=lab_labels).result()
sandbox.wait_for_running(timeout=120)

details = client.get_sandbox(sandbox.sandbox_id)
print(f'📦 Created sandbox from BYOC image: {sandbox.sandbox_id}  state={details.state}')

# Demonstrate the custom image is in use: the default sandbox disk is Ubuntu without
# Node.js installed, while our image is the Node sample — so `node --version` only
# succeeds when the sandbox actually booted from our bring-your-own-container disk.
print('\n🔎 Proving the custom image booted:')
node_version = sandbox.exec('node --version')
print(f'  🟢 node:   {node_version.stdout.strip()}  (exit={node_version.exit_code})')
print(f"  🐧 os:     {sandbox.exec('. /etc/os-release && echo $PRETTY_NAME').stdout.strip()}")
app_path = sandbox.exec('find / -name server.js 2>/dev/null | head -n1').stdout.strip()
print(f'  📄 app:    {app_path or "(server.js not found)"}')

assert node_version.exit_code == 0 and node_version.stdout.strip(), 'Node not found — the BYOC image did not boot as expected'
print('\n✅ Sandbox is running our bring-your-own-container image.')


### 7. Expose the app and prove it serves traffic

The `helloworld-node` image bundles a small web server. We publish a public HTTPS endpoint with `sandbox.add_port(...)`, print the URL, then `curl` it to confirm the bring-your-own-container app is actually serving requests.


In [ ]:
# The helloworld-node sample listens on port 80. Reuse the mapping if it already exists.
app_port = 80
port = next((p for p in sandbox.get().ports if p.port == app_port), None)
if port:
    print(f'♻️ Port {app_port} already exposed.')
else:
    port = sandbox.add_port(app_port, anonymous=True)
    print(f'✅ Exposed port {app_port}')
print(f'🌐 Public URL: {port.url}')

# Poll the public URL — can take a few seconds to be reachable.
last_err = None
for attempt in range(10):
    try:
        with urllib.request.urlopen(port.url, timeout=5) as resp:
            body = resp.read().decode()
        print(f'✅ HTTP {resp.status} (after {attempt + 1} attempt[s]) — content from the container: {body}')
        break
    except Exception as ex:
        last_err = ex
        time.sleep(1)
else:
    print(f'⚠️ Could not reach {port.url}: {last_err}')


### 8. Clean up

We use the **lab label** (`lab_labels`) we attached to every sandbox throughout this notebook to find and delete them in one pass — that way leftover sandboxes from a previous run (or a crashed kernel) get cleaned up too, not just the `sandbox` variable in scope.

Then we delete the **Azure Container Registry** we created, and the **sandbox group**, which cascades and removes any remaining sandboxes, disk images, snapshots, volumes, and secrets scoped to it.

The resource group delete is left commented out so you don't accidentally remove a shared RG — uncomment it if you created the RG just for this lab.


In [ ]:
# Delete the created sandbox.
client.delete_sandbox(sandbox.sandbox_id)
print(f'🗑️ Deleted sandbox: {sandbox.sandbox_id}')

# Delete the Azure Container Registry we created for this lab.
if 'acr_name' in globals():
    subprocess.run(
        f'az acr delete --name {acr_name} --resource-group {resource_group_name} --yes -o none',
        capture_output=True, text=True, shell=True)
    print(f'🗑️ Deleted Azure Container Registry: {acr_name}')

# Delete the sandbox group — removes any remaining sandboxes, disk images, snapshots, volumes, and secrets scoped to it.
sandboxgroup_mgmt_client.delete_group(sandbox_group_name)
print(f'🗑️ Deleted sandbox group: {sandbox_group_name}')

# Optionally delete the resource group (fire-and-forget — runs in the background).
# resource_mgmt_client.resource_groups.begin_delete(resource_group_name)
# print(f'🗑️ Deleting resource group (async): {resource_group_name}')
